# Notebook 05 — StratLake Q1 Feature Data Generation with Fintech Daily Bars

This notebook continues the latest Notebook 04 session/archive layout.

It keeps the Fintech and StratLake session identities separate:

```text
FINTECH_SESSION_ID   -> upstream curated market-data session
STRATLAKE_SESSION_ID -> downstream feature/research session
```

The feature build needs local curated daily bars first, so this notebook performs a lightweight Fintech daily-bars ingestion step and then runs StratLake feature generation against the explicit local `MARKETLAKE_ROOT`.

Q1 window used in this presentation:

```text
start = 2025-01-01
end   = 2025-04-01
```

The main flow is:

```text
Alpaca daily bars
        ↓
Fintech local curated data under MARKETLAKE_ROOT
        ↓
optional Fintech archive pack on Drive
        ↓
StratLake Q1 feature generation
        ↓
optional StratLake session export/archive checkpoint
```

Drive is persistence/restore storage only. The active ingestion and feature-generation workflow runs from local `/content` workspace paths.


## Install required presentation packages

Run this cell in a fresh notebook runtime.

This tutorial series uses `fintech-market-ingestion` as the source of curated market data and `stratlake-trade-engine` as the feature-generation and research platform.

In [ ]:
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine

## Verify required CLI commands

These notebooks use both `fintech-market-ingestion` and `stratlake-trade-engine` commands.

The latest storage/archive workflow expects both the session CLIs and archive/backup CLIs to be available.

If any command is missing, rerun the install cell above.


In [ ]:
import shutil

required_commands = [
    "fintech-init-project",
    "fintech-backfill-daily",
    "fintech-save-session",
    "fintech-restore-session",
    "fintech-backup-data",
    "stratlake-init-session",
    "stratlake-build-features",
    "stratlake-session-export",
    "stratlake-session-import",
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
]

for command in required_commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'NOT FOUND'}")


## Authorize Google Drive access

Run this cell in Google Colab to authorize Google Drive access.

The packages do not mount Google Drive for you. Drive access is user-initiated here, and later commands treat Drive as a mounted filesystem path.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Define shared Fintech and StratLake paths

The Fintech workspace provides curated market data.

The StratLake workspace consumes that curated data through an explicit `MARKETLAKE_ROOT` and generates feature data for research workflows.

Google Drive is used as persistence and archive storage only. The active runtime remains local-first for faster Colab execution.

The Drive layout mirrors Notebook 04:

```text
/content/drive/MyDrive/{DRIVE_FOLDER_NAME}/
  fintech-market-ingestion/sessions/{FINTECH_SESSION_ID}/
  stratlake-trade-engine/sessions/{STRATLAKE_SESSION_ID}/
```


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

FINTECH_ROOT = Path("/content/fintech-market-ingestion-demo")
STRATLAKE_ROOT = Path("/content/stratlake-trade-engine-demo")

DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )
FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"

# Keep names descriptive, but let the init commands own the final session ID.
# This preserves {SESSION_ID} discovery from manifests and avoids hardcoded Drive folders.
FINTECH_SESSION_NAME = f"fintech_stratlake_input_{TIMESTAMP_UTC}"
STRATLAKE_SESSION_NAME = f"stratlake_q1_features_{TIMESTAMP_UTC}"

print("FINTECH_ROOT:", FINTECH_ROOT)
print("STRATLAKE_ROOT:", STRATLAKE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("FINTECH_DRIVE_ROOT:", FINTECH_DRIVE_ROOT)
print("STRATLAKE_DRIVE_ROOT:", STRATLAKE_DRIVE_ROOT)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("FINTECH_SESSION_NAME:", FINTECH_SESSION_NAME)
print("STRATLAKE_SESSION_NAME:", STRATLAKE_SESSION_NAME)


## Initialize the Fintech project session

This session provides the curated input data location for StratLake.

The important rule is that downstream cells discover `FINTECH_SESSION_ID` from the session manifest rather than hardcoding a session folder.


In [ ]:
!fintech-init-project \
  --root {FINTECH_ROOT.as_posix()} \
  --notebooks \
  --with-session \
  --session-name {FINTECH_SESSION_NAME}

## Extract `FINTECH_SESSION_ID`

The Fintech session ID is read from the latest Fintech project-session manifest.

In [ ]:
fintech_manifest_paths = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime,
)

if not fintech_manifest_paths:
    raise FileNotFoundError("No Fintech session_manifest.json files found.")

FINTECH_SESSION_MANIFEST_PATH = fintech_manifest_paths[-1]
FINTECH_SESSION_MANIFEST = json.loads(FINTECH_SESSION_MANIFEST_PATH.read_text(encoding="utf-8"))
FINTECH_SESSION_ID = FINTECH_SESSION_MANIFEST["session_id"]

print("FINTECH_SESSION_MANIFEST_PATH:", FINTECH_SESSION_MANIFEST_PATH)
print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)

## Initialize the StratLake project session

StratLake uses an explicit session-first workspace that records the selected project root, external MarketLake root, and Drive persistence root.

`MARKETLAKE_ROOT` points to the Fintech curated-data directory, not to a copied StratLake-owned dataset. This keeps Fintech as the upstream data provider and StratLake as the feature/research consumer.

Use `--notebook-configs` to include the notebook-oriented config bundle, including `universe.yml` and `paths.yml`, with the session setup. The CLI does **not** accept arbitrary `--include-configs` or repeated `--config-file` arguments.


In [ ]:
# Initialize the StratLake notebook session and include notebook config files.
#
# Use --force-notebook-configs when re-running this cell and you want to refresh
# existing generated notebook config files.
!stratlake-init-session \
  --root {STRATLAKE_ROOT.as_posix()} \
  --project-name {STRATLAKE_SESSION_NAME} \
  --marketlake-root {MARKETLAKE_ROOT.as_posix()} \
  --drive-root {DRIVE_ROOT.as_posix()} \
  --enable-drive-persistence \
  --notebook-configs


## Verify notebook config files

Confirm that `universe.yml` and `paths.yml` are present after initialization.

These config files are part of the StratLake notebook session setup and should travel with the generated session config bundle.


In [ ]:
expected_notebook_configs = [
    STRATLAKE_ROOT / "configs" / "universe.yml",
    STRATLAKE_ROOT / "configs" / "paths.yml",
]

for config_path in expected_notebook_configs:
    print(f"{config_path}: {'FOUND' if config_path.exists() else 'MISSING'}")

missing_configs = [path for path in expected_notebook_configs if not path.exists()]
if missing_configs:
    raise FileNotFoundError(
        "Missing expected StratLake notebook config files: "
        + ", ".join(path.as_posix() for path in missing_configs)
    )


## Extract `STRATLAKE_SESSION_ID`

StratLake session metadata lives under `.stratlake/session.json`.

Some StratLake session metadata uses `project_name` as the stable identifier. This cell extracts `session_id` when present and otherwise falls back to `project_name`.

In [ ]:
STRATLAKE_SESSION_FILE = STRATLAKE_ROOT / ".stratlake" / "session.json"

if not STRATLAKE_SESSION_FILE.exists():
    raise FileNotFoundError(f"Missing StratLake session file: {STRATLAKE_SESSION_FILE}")

STRATLAKE_SESSION_MANIFEST = json.loads(STRATLAKE_SESSION_FILE.read_text(encoding="utf-8"))
STRATLAKE_SESSION_ID = (
    STRATLAKE_SESSION_MANIFEST.get("session_id")
    or STRATLAKE_SESSION_MANIFEST.get("project_name")
    or STRATLAKE_SESSION_NAME
)

print("STRATLAKE_SESSION_FILE:", STRATLAKE_SESSION_FILE)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)

## Create SESSION_ID-based Google Drive folders and archive IDs

This cell creates Drive folders only if they do not already exist.

Both Fintech and StratLake get session-scoped persistence folders:

```text
fintech-market-ingestion/sessions/{FINTECH_SESSION_ID}
stratlake-trade-engine/sessions/{STRATLAKE_SESSION_ID}
```

Archive IDs are also derived from active session IDs:

```text
FINTECH_ARCHIVE_ID   = curated-data-{FINTECH_SESSION_ID}
STRATLAKE_ARCHIVE_ID = stratlake-session-{STRATLAKE_SESSION_ID}
```

These archive IDs are transfer/restore identifiers, not canonical data sources.


In [ ]:
if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError(
        "Set DRIVE_FOLDER_NAME before creating Google Drive session/archive folders."
    )

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Google Drive before creating session/archive folders.")

FINTECH_DRIVE_SESSIONS_ROOT = FINTECH_DRIVE_ROOT / "sessions"
STRATLAKE_DRIVE_SESSIONS_ROOT = STRATLAKE_DRIVE_ROOT / "sessions"

FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / FINTECH_SESSION_ID
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / STRATLAKE_SESSION_ID

FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSION_ROOT / "backups"
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"

FINTECH_BACKUP_PACK_DIR = FINTECH_DRIVE_BACKUP_ROOT / FINTECH_ARCHIVE_ID
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

for path in [
    FINTECH_DRIVE_SESSION_ROOT,
    STRATLAKE_DRIVE_SESSION_ROOT,
    FINTECH_DRIVE_BACKUP_ROOT,
    STRATLAKE_DRIVE_ARCHIVE_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

FINTECH_ROOT_STR = FINTECH_ROOT.as_posix()
STRATLAKE_ROOT_STR = STRATLAKE_ROOT.as_posix()
MARKETLAKE_ROOT_STR = MARKETLAKE_ROOT.as_posix()
DRIVE_ROOT_STR = DRIVE_ROOT.as_posix()

FINTECH_DRIVE_ROOT_STR = FINTECH_DRIVE_ROOT.as_posix()
STRATLAKE_DRIVE_ROOT_STR = STRATLAKE_DRIVE_ROOT.as_posix()
FINTECH_DRIVE_SESSION_ROOT_STR = FINTECH_DRIVE_SESSION_ROOT.as_posix()
STRATLAKE_DRIVE_SESSION_ROOT_STR = STRATLAKE_DRIVE_SESSION_ROOT.as_posix()
FINTECH_DRIVE_BACKUP_ROOT_STR = FINTECH_DRIVE_BACKUP_ROOT.as_posix()
STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()
FINTECH_BACKUP_PACK_DIR_STR = FINTECH_BACKUP_PACK_DIR.as_posix()
STRATLAKE_ARCHIVE_PACK_DIR_STR = STRATLAKE_ARCHIVE_PACK_DIR.as_posix()

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("FINTECH_ARCHIVE_ID:", FINTECH_ARCHIVE_ID)
print("STRATLAKE_ARCHIVE_ID:", STRATLAKE_ARCHIVE_ID)
print("FINTECH_DRIVE_SESSION_ROOT:", FINTECH_DRIVE_SESSION_ROOT)
print("STRATLAKE_DRIVE_SESSION_ROOT:", STRATLAKE_DRIVE_SESSION_ROOT)
print("FINTECH_DRIVE_BACKUP_ROOT:", FINTECH_DRIVE_BACKUP_ROOT)
print("STRATLAKE_DRIVE_ARCHIVE_ROOT:", STRATLAKE_DRIVE_ARCHIVE_ROOT)
print("FINTECH_BACKUP_PACK_DIR:", FINTECH_BACKUP_PACK_DIR)
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR)


## Optional: choose previous Drive sessions for restore

Leave these values set to the current session IDs unless you intentionally want to restore from an older Drive session.

For a previous Fintech session, update both:

```python
RESTORE_FINTECH_SESSION_ID = "..."
RESTORE_FINTECH_ARCHIVE_ID = f"curated-data-{RESTORE_FINTECH_SESSION_ID}"
```

The default keeps restore previews tied to the current `{FINTECH_SESSION_ID}`.


In [ ]:
available_fintech_sessions = sorted(
    path.name for path in FINTECH_DRIVE_SESSIONS_ROOT.glob("*") if path.is_dir()
)
available_stratlake_sessions = sorted(
    path.name for path in STRATLAKE_DRIVE_SESSIONS_ROOT.glob("*") if path.is_dir()
)

print("Available Fintech Drive sessions:")
for session_id in available_fintech_sessions:
    print(" -", session_id)

print("\nAvailable StratLake Drive sessions:")
for session_id in available_stratlake_sessions:
    print(" -", session_id)

RESTORE_FINTECH_SESSION_ID = FINTECH_SESSION_ID
RESTORE_STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID

RESTORE_FINTECH_ARCHIVE_ID = f"curated-data-{RESTORE_FINTECH_SESSION_ID}"
RESTORE_STRATLAKE_ARCHIVE_ID = f"stratlake-session-{RESTORE_STRATLAKE_SESSION_ID}"

RESTORE_FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / RESTORE_FINTECH_SESSION_ID
RESTORE_STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / RESTORE_STRATLAKE_SESSION_ID

RESTORE_FINTECH_DRIVE_BACKUP_ROOT = RESTORE_FINTECH_DRIVE_SESSION_ROOT / "backups"
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT = RESTORE_STRATLAKE_DRIVE_SESSION_ROOT / "archives"

RESTORE_FINTECH_BACKUP_PACK_DIR = RESTORE_FINTECH_DRIVE_BACKUP_ROOT / RESTORE_FINTECH_ARCHIVE_ID
RESTORE_STRATLAKE_ARCHIVE_PACK_DIR = RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT / RESTORE_STRATLAKE_ARCHIVE_ID

RESTORE_FINTECH_DRIVE_SESSION_ROOT_STR = RESTORE_FINTECH_DRIVE_SESSION_ROOT.as_posix()
RESTORE_STRATLAKE_DRIVE_SESSION_ROOT_STR = RESTORE_STRATLAKE_DRIVE_SESSION_ROOT.as_posix()
RESTORE_FINTECH_DRIVE_BACKUP_ROOT_STR = RESTORE_FINTECH_DRIVE_BACKUP_ROOT.as_posix()
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()
RESTORE_FINTECH_BACKUP_PACK_DIR_STR = RESTORE_FINTECH_BACKUP_PACK_DIR.as_posix()
RESTORE_STRATLAKE_ARCHIVE_PACK_DIR_STR = RESTORE_STRATLAKE_ARCHIVE_PACK_DIR.as_posix()

print("\nRESTORE_FINTECH_SESSION_ID:", RESTORE_FINTECH_SESSION_ID)
print("RESTORE_FINTECH_ARCHIVE_ID:", RESTORE_FINTECH_ARCHIVE_ID)
print("RESTORE_STRATLAKE_SESSION_ID:", RESTORE_STRATLAKE_SESSION_ID)
print("RESTORE_STRATLAKE_ARCHIVE_ID:", RESTORE_STRATLAKE_ARCHIVE_ID)
print("Restore Fintech session path exists:", RESTORE_FINTECH_DRIVE_SESSION_ROOT.exists())
print("Restore Fintech archive pack exists:", RESTORE_FINTECH_BACKUP_PACK_DIR.exists())
print("Restore StratLake session path exists:", RESTORE_STRATLAKE_DRIVE_SESSION_ROOT.exists())
print("Restore StratLake archive pack exists:", RESTORE_STRATLAKE_ARCHIVE_PACK_DIR.exists())


## Prepare a StratLake ticker file

This file selects a small demonstration universe for Q1 feature generation.

In [ ]:
STRATLAKE_CONFIGS_ROOT = STRATLAKE_ROOT / "configs"
STRATLAKE_CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)

STRATLAKE_TICKERS_FILE = STRATLAKE_CONFIGS_ROOT / "tickers_q1_demo.txt"
STRATLAKE_TICKERS_FILE.write_text("AAPL\nMSFT\nNVDA\n", encoding="utf-8")

STRATLAKE_TICKERS_FILE_STR = STRATLAKE_TICKERS_FILE.as_posix()

print("StratLake tickers file:", STRATLAKE_TICKERS_FILE)
print(STRATLAKE_TICKERS_FILE.read_text(encoding="utf-8"))

## Configure Alpaca API credentials from Colab Secrets

Before StratLake can generate features, this tutorial needs Q1 daily bars available in the local Fintech curated-data workspace.

This cell reads Alpaca credentials from **Colab Secrets** when available and falls back to a hidden prompt if needed.

Recommended Colab Secrets:

```text
ALPACA_API_KEY_ID
ALPACA_API_SECRET_KEY
```

The API key and secret are not printed.


In [ ]:
import os
import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value

alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

if not alpaca_api_key_id or not alpaca_api_secret_key:
    raise ValueError("Missing Alpaca API credentials.")

os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
os.environ["ALPACA_FEED"] = "iex"

print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set (hidden for security).")


## Prepare the Fintech ticker file and daily bars output path

StratLake consumes curated market data from the Fintech workspace through `MARKETLAKE_ROOT`.

This cell prepares a small Q1 demonstration ticker file and defines the local daily bars output path.

In [ ]:
FINTECH_CONFIGS_ROOT = FINTECH_ROOT / "configs"
FINTECH_CONFIGS_ROOT.mkdir(parents=True, exist_ok=True)

FINTECH_TICKERS_FILE = FINTECH_CONFIGS_ROOT / "tickers_q1_demo.txt"
FINTECH_TICKERS_FILE.write_text("AAPL\nMSFT\nNVDA\n", encoding="utf-8")

DAILY_BARS_ROOT = MARKETLAKE_ROOT / "bars_daily"
DAILY_BARS_ROOT.mkdir(parents=True, exist_ok=True)

FINTECH_TICKERS_FILE_STR = FINTECH_TICKERS_FILE.as_posix()
DAILY_BARS_ROOT_STR = DAILY_BARS_ROOT.as_posix()

print("Fintech tickers file:", FINTECH_TICKERS_FILE)
print(FINTECH_TICKERS_FILE.read_text(encoding="utf-8"))
print("Daily bars output root:", DAILY_BARS_ROOT)

## Optional: restore Fintech curated data before ingestion

Use this only when you already have a previous Fintech archive pack on Drive and want to avoid pulling daily bars again.

The default notebook path is still local-first: restore into local `MARKETLAKE_ROOT`, then run StratLake from local files.


In [ ]:
print("Current MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("Current MARKETLAKE_ROOT exists:", MARKETLAKE_ROOT.exists())
print("Restore Fintech archive pack:", RESTORE_FINTECH_BACKUP_PACK_DIR)
print("Restore pack exists:", RESTORE_FINTECH_BACKUP_PACK_DIR.exists())

# Example restore command, intentionally commented.
# Update RESTORE_FINTECH_SESSION_ID / RESTORE_FINTECH_ARCHIVE_ID above before using this.
#
# !fintech-backup-data restore \
#   --root {FINTECH_ROOT_STR} \
#   --archive-id {RESTORE_FINTECH_ARCHIVE_ID} \
#   --drive-root {RESTORE_FINTECH_DRIVE_BACKUP_ROOT_STR} \
#   --target-root {MARKETLAKE_ROOT_STR} \
#   --copy-policy overwrite_allowed \
#   --validate-after-copy \
#   --inspect-after-copy


## Extract Q1 daily bars into the local Fintech curated-data workspace

This command pulls daily bars from Alpaca into:

```text
/content/fintech-market-ingestion-demo/data/curated/bars_daily
```

StratLake will use this local curated-data root as its external `MARKETLAKE_ROOT`.

The command uses `{FINTECH_SESSION_ID}` in the `--source` tag so the extraction can be associated with the current Fintech notebook session.

In [ ]:
!fintech-backfill-daily \
  --symbols {FINTECH_TICKERS_FILE_STR} \
  --start 2025-01-01 \
  --end 2025-04-01 \
  --out {DAILY_BARS_ROOT_STR} \
  --feed iex \
  --source session_{FINTECH_SESSION_ID} \
  --window month

## Inspect extracted Q1 daily bars

Confirm that local curated daily bars exist before running the StratLake feature build.

In [ ]:
daily_bar_files = sorted(DAILY_BARS_ROOT.rglob("*.parquet"))

print("Daily bars root exists:", DAILY_BARS_ROOT.exists())
print("Daily bars parquet file count:", len(daily_bar_files))

for path in daily_bar_files[:20]:
    print(path)

## Precheck the upstream curated-data root

This notebook expects curated market data to exist under the Fintech workspace.

If this check finds no Parquet files, run the earlier Fintech extraction notebook or restore a previously saved/archived Fintech dataset before building StratLake features.

In [ ]:
MARKETLAKE_PARQUET_FILES = sorted(MARKETLAKE_ROOT.rglob("*.parquet"))

print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("MARKETLAKE_ROOT exists:", MARKETLAKE_ROOT.exists())
print("Parquet file count:", len(MARKETLAKE_PARQUET_FILES))

for path in MARKETLAKE_PARQUET_FILES[:20]:
    print(path)

## Optional: archive the Fintech Q1 curated-data input

After daily bars are available locally, you can checkpoint the upstream Fintech curated data into a `{FINTECH_SESSION_ID}`-scoped Drive archive pack.

This is useful when you want to skip a future API pull and restore the same Q1 input dataset into a fresh Colab runtime.

Archive packs are transfer/restore artifacts. They are not the active dataset and should not replace `MARKETLAKE_ROOT` as the local input path.


In [ ]:
fintech_pack_preview = f'''
fintech-backup-data pack \
  --root {FINTECH_ROOT_STR} \
  --dataset-root {MARKETLAKE_ROOT_STR} \
  --archive-id {FINTECH_ARCHIVE_ID} \
  --drive-root {FINTECH_DRIVE_BACKUP_ROOT_STR} \
  --copy-policy overwrite_allowed \
  --validate-after-copy \
  --inspect-after-copy
'''.strip()

print("Fintech archive pack preview:")
print(fintech_pack_preview)

# Uncomment to create the archive pack after checking the preview above.
#
# !fintech-backup-data pack \
#   --root {FINTECH_ROOT_STR} \
#   --dataset-root {MARKETLAKE_ROOT_STR} \
#   --archive-id {FINTECH_ARCHIVE_ID} \
#   --drive-root {FINTECH_DRIVE_BACKUP_ROOT_STR} \
#   --copy-policy overwrite_allowed \
#   --validate-after-copy \
#   --inspect-after-copy


## Build Q1 daily features with StratLake

This command builds daily features from the external curated market-data root.

The explicit `--marketlake-root` keeps the notebook current working directory from controlling where input data is read from.

The cell also switches the runtime working directory to `STRATLAKE_ROOT` before invoking the CLI so relative StratLake outputs land in the intended workspace.


In [ ]:
import os

os.chdir(STRATLAKE_ROOT)
print("Current working directory:", Path.cwd())
print("Using MARKETLAKE_ROOT:", MARKETLAKE_ROOT_STR)

!stratlake-build-features \
  --timeframe 1D \
  --start 2025-01-01 \
  --end 2025-04-01 \
  --tickers {STRATLAKE_TICKERS_FILE_STR} \
  --marketlake-root {MARKETLAKE_ROOT_STR}


## Inspect generated StratLake feature files

This cell searches the StratLake workspace for generated feature files.

In [ ]:
feature_candidates = sorted((STRATLAKE_ROOT / "data").rglob("*.parquet"))

print("Generated/available StratLake parquet files:", len(feature_candidates))
for path in feature_candidates[:30]:
    print(path)

## Optional: export the feature session snapshot to Drive

This dry run previews a StratLake session export that includes features, artifacts, and configs.

It uses the session-scoped StratLake Drive root:

```text
stratlake-trade-engine/sessions/{STRATLAKE_SESSION_ID}
```


In [ ]:
!stratlake-session-export \
  --root {STRATLAKE_ROOT_STR} \
  --drive-root {STRATLAKE_DRIVE_SESSION_ROOT_STR} \
  --include-features \
  --include-artifacts \
  --include-configs \
  --dry-run


## Optional: archive the StratLake feature session

For a larger or more portable checkpoint, use the StratLake archive bootstrap workflow after the feature build.

This archive is scoped to `{STRATLAKE_SESSION_ID}` and should include generated features, artifacts, and configs. Because Notebook 04/05 initialize with `--notebook-configs`, the config bundle should include notebook-oriented files such as `universe.yml` and `paths.yml`.

The archive remains a Drive-backed transfer/restore artifact, not a second source of truth.


In [ ]:
stratlake_archive_preview = f'''
stratlake-session-archive-bootstrap \
  --root {STRATLAKE_ROOT_STR} \
  --archive-id {STRATLAKE_ARCHIVE_ID} \
  --archive-collision-policy overwrite_allowed \
  --drive-root {STRATLAKE_DRIVE_ARCHIVE_ROOT_STR} \
  --copy-policy overwrite_allowed \
  --include-features \
  --include-artifacts \
  --include-configs \
  --validate-after-copy \
  --inspect-after-copy
'''.strip()

stratlake_restore_preview = f'''
stratlake-session-archive-restore-bootstrap \
  --root {STRATLAKE_ROOT_STR} \
  --archive-id {RESTORE_STRATLAKE_ARCHIVE_ID} \
  --drive-root {RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT_STR} \
  --copy-policy overwrite_allowed \
  --include-features \
  --include-artifacts \
  --include-configs \
  --validate-after-copy \
  --inspect-after-copy
'''.strip()

print("StratLake archive bootstrap preview:")
print(stratlake_archive_preview)
print("\nStratLake archive restore preview:")
print(stratlake_restore_preview)

# Uncomment to create the StratLake archive after checking the preview above.
#
# !stratlake-session-archive-bootstrap \
#   --root {STRATLAKE_ROOT_STR} \
#   --archive-id {STRATLAKE_ARCHIVE_ID} \
#   --archive-collision-policy overwrite_allowed \
#   --drive-root {STRATLAKE_DRIVE_ARCHIVE_ROOT_STR} \
#   --copy-policy overwrite_allowed \
#   --include-features \
#   --include-artifacts \
#   --include-configs \
#   --validate-after-copy \
#   --inspect-after-copy


## Notebook summary

Notebook 05 generated StratLake Q1 feature data while preserving the latest session/archive conventions from Notebook 04.

You:

- initialized a Fintech project session and discovered `FINTECH_SESSION_ID` from the manifest,
- initialized a StratLake notebook session with `--notebook-configs`,
- verified that `configs/universe.yml` and `configs/paths.yml` exist,
- preserved separate `FINTECH_SESSION_ID` and `STRATLAKE_SESSION_ID` values,
- created session-scoped Drive paths under:

```text
fintech-market-ingestion/sessions/{FINTECH_SESSION_ID}
stratlake-trade-engine/sessions/{STRATLAKE_SESSION_ID}
```

- derived archive IDs from the active session IDs,
- optionally restored Fintech curated data from a previous archive,
- pulled Q1 daily bars into local `MARKETLAKE_ROOT`,
- optionally prepared a Fintech archive pack for the Q1 curated input,
- built StratLake daily features from the explicit Fintech `MARKETLAKE_ROOT`,
- previewed StratLake session export/archive commands that include features, artifacts, and configs.

Boundary rule:

```text
Fintech curated data archive ≠ StratLake feature archive
Fintech session ID ≠ StratLake session ID
Drive backup/archive packs ≠ canonical active workspace data
MARKETLAKE_ROOT remains the explicit handoff from Fintech to StratLake
```
